# 📄 Notebook 02 — Text & Document Navigation

**What you'll learn:**
- Handle producers that return raw text (HTML, PDF, logs, markdown)
- Use `text_content` toolset for char-based pagination and in-text search
- Use transforms (`chunk_text`, `split_sections`, `split_markdown_sections`) to convert text → structured data
- Shape-aware tool filtering: the LLM only sees compatible tools

**Real-world use cases:**
- Web scraping agent (fetch URL → navigate the HTML/text)
- Document analysis (PDF → searchable chunks)
- Log file investigation (huge log → paginate + search)

## 1. The Problem: Tools That Return Strings

In [2]:
# A webpage scraper returns a huge string. What does the LLM do with it?
sample_html = "<!DOCTYPE html>\n<html>\n" + "\n".join(
    f"<p>Paragraph {i}: Lorem ipsum dolor sit amet, consectetur adipiscing elit. "
    f"Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua.</p>"
    for i in range(200)
) + "\n</html>"

print(f"Document size: {len(sample_html):,} chars ({len(sample_html) // 4:,} tokens)")
print("Stuffing this into context wastes your token budget on boilerplate HTML.")

Document size: 29,120 chars (7,280 tokens)
Stuffing this into context wastes your token budget on boilerplate HTML.


## 2. Solution A: `text_content` — Navigate Text In Place

In [3]:
from ctxtual import Forge, MemoryStore
from ctxtual.utils import text_content
import json

forge = Forge(store=MemoryStore())
reader = text_content(forge, "page")

@forge.producer(workspace_type="page", toolsets=[reader])
def fetch_webpage(url: str) -> str:
    """Fetch a webpage and return its text content."""
    # Simulated — in production, use requests/httpx/playwright
    return "<!DOCTYPE html>\n<html>\n" + "\n".join(
        f"<p>Section {i}: This discusses topic #{i} about artificial intelligence "
        f"and its applications in healthcare, finance, and education. "
        f"Key findings include improved accuracy by {i * 2.5}% over baseline.</p>"
        for i in range(300)
    ) + "\n</html>"

ref = fetch_webpage("https://example.com/ai-report")
print(json.dumps(ref, indent=2))

{
  "status": "workspace_ready",
  "workspace_id": "page_f5bd247d3f",
  "workspace_type": "page",
  "item_count": 1,
  "message": "\u2713 1 item(s) stored in workspace 'page_f5bd247d3f' (type: page). Use the tools below to explore the data.",
  "available_tools": [
    "page_read_page(workspace_id='page_f5bd247d3f')",
    "page_search_in_text(workspace_id='page_f5bd247d3f')",
    "page_get_length(workspace_id='page_f5bd247d3f')"
  ],
  "data_shape": "scalar",
  "next_steps": [
    "\u2022 page_read_page: Read a page of text from a document workspace",
    "\u2022 page_search_in_text: Search for text within a document and return matches with surrounding context",
    "\u2022 page_get_length: Return character count, word count, and line count of the document"
  ]
}


👆 Notice:
- `data_shape: "scalar"` — the library knows it's a string, not a list
- `item_count: 1` — one document (not the character count!)
- Only `text_content` tools appear — no paginator/filter (they'd break on strings)
- No `item_schema` (scalars don't have fields)

In [ ]:
workspace_id = ref["workspace_id"]

# Read first page (4000 chars by default)
page = forge.dispatch_tool_call("page_read_page", {
    "workspace_id": workspace_id,
    "page": 0,
    "chars_per_page": 2000,
})
data = page["result"]
print(f"Page {data['page']} of {data['total_pages']}")
print(f"Has next: {data['has_next']}")
print(f"\nContent preview:\n{data['text'][:200]}...")

In [ ]:
# Search within the text
search_result = forge.dispatch_tool_call("page_search_in_text", {
    "workspace_id": workspace_id,
    "query": "healthcare",
    "context_chars": 80,
    "max_results": 5,
})
print(f"Found {search_result['match_count']} occurrences of 'healthcare'\n")
for match in search_result["matches"][:3]:
    print(f"  Offset {match['char_offset']}: ...{match['context']}...")
    print()

In [ ]:
# Get document stats
length = forge.dispatch_tool_call("page_get_length", {
    "workspace_id": workspace_id,
})
print(f"Characters: {length['chars']:,}")
print(f"Words:      {length['words']:,}")
print(f"Lines:      {length['lines']:,}")

## 3. Solution B: Transforms — Convert Text to Structured Data

When you want to use **list-based tools** (paginate, filter, pipeline) with text,
transform it before storing. This is the better approach when:
- You need to paginate by semantic chunks (not raw chars)
- You want to search/filter individual sections
- You're building a RAG pipeline

### 3a. `chunk_text` — Fixed-Size Overlapping Chunks

In [ ]:
from ctxtual import chunk_text
from ctxtual.utils import paginator, text_search, pipeline

forge2 = Forge(store=MemoryStore())
pager = paginator(forge2, "chunks")
search = text_search(forge2, "chunks", fields=["text"])
pipe = pipeline(forge2, "chunks")

# chunk_text() is a TRANSFORM FACTORY — pass it to the producer decorator.
# The producer returns a raw string; chunk_text splits it into overlapping dicts.
@forge2.producer(
    workspace_type="chunks",
    toolsets=[pager, search, pipe],
    transform=chunk_text(chunk_size=500, overlap=50),
)
def ingest_document(content: str) -> str:
    """Ingest a document — the chunk_text transform splits it automatically."""
    return content

# Simulate a long document
long_doc = "\n".join(
    f"Chapter {i}: This chapter covers topic {i} in depth. "
    f"We explore the implications of finding #{i} and how it relates to the broader field. "
    f"The key insight is that method {i} outperforms baseline by {i * 1.5:.1f}%."
    for i in range(100)
)

ref = ingest_document(long_doc)
print(f"Document chunked into {ref['item_count']} overlapping chunks")
print(f"Schema: {json.dumps(ref.get('item_schema', {}), indent=2)}")

In [ ]:
# Now you can paginate chunks, search them, even pipeline them
wid = ref["workspace_id"]

# Search across chunks
results = forge2.dispatch_tool_call("chunks_search", {
    "workspace_id": wid,
    "query": "chapter 42",
    "max_results": 3,
})
print("=== Search in chunks: 'chapter 42' ===")
for item in results["matches"]:
    print(f"  Chunk {item['chunk_index']}: {item['text'][:80]}...")

# Pipeline: find chunks mentioning "chapter 5" and sort
pipe_result = forge2.dispatch_tool_call("chunks_pipe", {
    "workspace_id": wid,
    "steps": [
        {"search": {"query": "chapter 5"}},
        {"limit": 3},
        {"select": ["chunk_index", "text"]},
    ],
})
print(f"\n=== Pipeline result: {len(pipe_result['items'])} chunks ===")
for item in pipe_result["items"]:
    print(f"  Chunk {item['chunk_index']}: {item['text'][:60]}...")

### 3b. `split_markdown_sections` — Structure-Aware Splitting

In [ ]:
from ctxtual import split_markdown_sections

forge3 = Forge(store=MemoryStore())
pager3 = paginator(forge3, "docs")
search3 = text_search(forge3, "docs", fields=["heading", "text"])
pipe3 = pipeline(forge3, "docs")

# split_markdown_sections() is a transform factory — pass to the decorator
@forge3.producer(
    workspace_type="docs",
    toolsets=[pager3, search3, pipe3],
    transform=split_markdown_sections(),
)
def parse_documentation(content: str) -> str:
    """Parse markdown documentation into navigable sections."""
    return content

markdown_doc = """# User Guide

Welcome to the product documentation.

## Installation

Run `pip install mypackage` to install. Requires Python 3.9+.

### From Source

Clone the repo and run `pip install -e .` for development mode.

## Configuration

Create a `config.yaml` file in your project root.

### Database Settings

Set `db.url` to your connection string. Supported: PostgreSQL, SQLite, MySQL.

### Cache Settings

Enable Redis caching with `cache.backend: redis` and `cache.url: redis://localhost`.

## API Reference

### Authentication

All endpoints require a Bearer token in the Authorization header.

### Endpoints

#### GET /users

Returns paginated list of users. Supports `?page=N&size=M` query params.

#### POST /users

Create a new user. Body: `{"name": "...", "email": "..."}`.

## Troubleshooting

### Common Errors

If you see `ConnectionRefused`, check that the database is running.
"""

ref = parse_documentation(markdown_doc)
print(f"Parsed into {ref['item_count']} sections")
print(f"Schema: {json.dumps(ref.get('item_schema', {}), indent=2)}")

In [ ]:
wid = ref["workspace_id"]

# Browse all sections
page = forge3.dispatch_tool_call("docs_paginate", {
    "workspace_id": wid,
    "page": 0,
    "size": 20,
})
print("=== All Sections ===")
for item in page["result"]["items"]:
    indent = "  " * (item["level"] - 1)
    print(f"{indent}{'#' * item['level']} {item['heading']}  ({len(item['text'])} chars)")

# Pipeline: find all level-2 headings
result = forge3.dispatch_tool_call("docs_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {"level": 2}},
        {"select": ["heading", "level"]},
    ],
})
print(f"\n=== Level 2 headings ===")
for item in result["items"]:
    print(f"  ## {item['heading']}")

### 3c. `split_sections` — Custom Delimiter Splitting

In [ ]:
from ctxtual import split_sections

# Split a log file by blank lines (each entry is a block)
log_file = """[2024-01-15 10:23:45] ERROR Connection timeout to db-primary
Stack trace: ConnectionError at pool.py:142
Retrying in 5s...

[2024-01-15 10:23:50] INFO Reconnected to db-primary
Pool size: 10, active: 3

[2024-01-15 10:24:01] WARN Slow query detected (2.3s)
Query: SELECT * FROM orders WHERE status = 'pending'
Table scan on 1.2M rows

[2024-01-15 10:24:15] ERROR Out of memory: heap allocation failed
Process RSS: 3.8GB, limit: 4GB
Initiating graceful shutdown...""".strip()

# split_sections() is a factory — call it to get the transform, then apply
splitter = split_sections(separator="\n\n")
sections = splitter(log_file)
print(f"Split into {len(sections)} log entries:")
for s in sections:
    print(f"  [{s['section_index']}] {s['text'][:60]}...")

## 4. Shape-Aware Tool Filtering

When you register both `text_content` (scalar-shape) and `paginator` (list-shape)
toolsets on the same workspace type, ctxtual exposes **all tools** but
automatically filters `available_tools` in the notification based on the *actual*
data shape returned by the producer.

The LLM only sees tools that will work with the data it received.

In [4]:
from ctxtual.utils import text_content, paginator
forge4 = Forge(store=MemoryStore())

# Register BOTH scalar and list toolsets on the same workspace type
reader4 = text_content(forge4, "mixed")
pager4 = paginator(forge4, "mixed")

# Producer that can return either shape
@forge4.producer(workspace_type="mixed", toolsets=[reader4, pager4])
def fetch_content(as_text: bool = True):
    """Fetch content — returns text or structured data depending on source."""
    if as_text:
        return "Hello, this is raw text content that could be a scraped webpage."
    else:
        return [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]

# Return a string → only text tools visible
ref_text = fetch_content(as_text=True)
print("=== String data → available tools ===")
for t in ref_text["available_tools"]:
    print(f"  {t}")

print()

# Return a list → only list tools visible
ref_list = fetch_content(as_text=False)
print("=== List data → available tools ===")
for t in ref_list["available_tools"]:
    print(f"  {t}")

print()
print("Shape-aware filtering: LLM only sees tools that match the actual data type")

=== String data → available tools ===
  mixed_read_page(workspace_id='mixed_340863db34')
  mixed_search_in_text(workspace_id='mixed_340863db34')
  mixed_get_length(workspace_id='mixed_340863db34')

=== List data → available tools ===
  mixed_paginate(workspace_id='mixed_959e7187e3')
  mixed_count(workspace_id='mixed_959e7187e3')
  mixed_get_item(workspace_id='mixed_959e7187e3')
  mixed_get_slice(workspace_id='mixed_959e7187e3')

Shape-aware filtering: LLM only sees tools that match the actual data type


## Summary

| Approach | Best For | Tools Available |
|----------|----------|-----------------|
| `text_content` | Raw text navigation (read page by page) | `read_page`, `search_in_text`, `get_length` |
| `chunk_text` | Fixed-size chunks for RAG/embeddings | All list tools (paginate, search, filter, pipeline) |
| `split_markdown_sections` | Structured docs with headers | All list tools + heading/level filtering |
| `split_sections` | Log files, delimiter-separated data | All list tools |

**Key insight:** ctxtual handles strings natively. You choose the strategy:
- Keep it as text → `text_content` for char-based navigation
- Transform it → `chunk_text` / `split_*` for structured exploration

**Next:** [03_pipelines_deep_dive.ipynb](03_pipelines_deep_dive.ipynb) — compound queries in a single tool call